In [1]:
# Import libraries
import pandas as pd
import re
import contractions

In [2]:
# Load dataset
df = pd.read_csv("../preparation/englishverses.csv", encoding='utf-8-sig')
print("Initial shape:", df.shape)


Initial shape: (44915, 17)


In [3]:
# Remove songs that have "white noise", "sleep" in genre
df = df[~df['artist_genres'].str.contains("native american music", case=False, na=False)]
print("After removing:", df.shape)

After removing: (44903, 17)


In [4]:
# Remove lyrics with unkown script
def contains_unknown_script(text):
    # Check for characters outside the basic Latin and common punctuation
    return bool(re.search(r'[^\x00-\x7F]', text))

df = df[~df['genius_lyrics'].apply(contains_unknown_script)]
print("Shape after removing unknown script:", df.shape)

Shape after removing unknown script: (40707, 17)


In [5]:
# Remove duplicates lyrics
df.drop_duplicates(subset='genius_lyrics', inplace=True)
print("Shape after removing duplicates:", df.shape)


Shape after removing duplicates: (34982, 17)


In [6]:
# Remove rows where 'lyrics' is NaN after cleaning
df = df.dropna(subset=["genius_lyrics"])
print("Shape after dropping NaN:", df.shape)

Shape after dropping NaN: (34982, 17)


In [7]:
# Check if column has square brackets
def has_square_brackets(text):
    return bool(re.search(r"\[.*?\]", text))

count = 0

for index, row in df.iterrows():
    if has_square_brackets(row['genius_lyrics']):
        print(f"Row {index} has square brackets: {row['genius_lyrics']}")
    else:
        count += 1

print(f"Number of rows without square brackets: {count}")

Number of rows without square brackets: 34982


In [8]:
# Clean lyrics function
def clean_lyrics(text):
    if not isinstance(text, str):
        return ""

    # Normalize apostrophes
    text = text.replace("’", "'")

    # Convert adlibs: (adlib) → , adlib,
    text = re.sub(r"\s*\((.*?)\)", r", \1,", text)

    # Fix merged words
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)

    # Apply contractions (easier for model training)
    try:
        text = contractions.fix(text)
    except:
        pass

    # Remove double commas
    text = re.sub(r",\s*,+", ", ", text)

    # Clean spaces around commas
    text = re.sub(r"\s+,", ",", text)
    text = re.sub(r",\s+", ", ", text)

    # Remove commas at the start of lines
    text = re.sub(r"^\s*,", "", text, flags=re.MULTILINE)

    # Remove double commas that are side by side
    text = re.sub(r",\s*,", ", ", text)

    # Remove trailing commas in each line
    text = re.sub(r",\s*$", "", text, flags=re.MULTILINE)

    return text.strip()

# Apply cleaning
df['lyrics'] = df['genius_lyrics'].apply(clean_lyrics)

In [9]:
# Remove genius_lyrics columns
df = df.drop(columns=['genius_lyrics'])

In [10]:
# Loop through dataset and save words that end with n' followed by space to a list
end_with_n_apostrophe = []

for index, row in df.iterrows():
    contractions_in_row = re.findall(r"\b\w+in' \b", row['lyrics'])
    end_with_n_apostrophe.extend(contractions_in_row)

print("Words that end with n':", set(end_with_n_apostrophe))
print("Size of list:", len(set(end_with_n_apostrophe)))

Words that end with n': {"namin' ", "preachin' ", "Defyin' ", "Playin' ", "soundin' ", "admittin' ", "fearin' ", "wigglin' ", "compromisin' ", "Sniffin' ", "givin' ", "exposin' ", "flossin' ", "wastin' ", "bleedin' ", "Showin' ", "Arguin' ", "Usin' ", "Lookin' ", "Meetin' ", "Stickin' ", "hittin' ", "chainin' ", "Swangin' ", "Nuttin' ", "sobbin' ", "stackin' ", "Headin' ", "Uppin' ", "Shinin' ", "feedin' ", "scrapin' ", "Coppin' ", "Tweakin' ", "respectin' ", "swipin' ", "Checkin' ", "mashin' ", "everythin' ", "borin' ", "Hosin' ", "Slippin' ", "robbin' ", "Trippin' ", "stealin' ", "Takin' ", "pushin' ", "Sprayin' ", "catchin' ", "clockin' ", "Watchin' ", "Admirin' ", "streamin' ", "movin' ", "mannin' ", "Clubbin' ", "Everythin' ", "blamin' ", "joinin' ", "pressin' ", "bathin' ", "knockin' ", "dodgin' ", "drivin' ", "retreatin' ", "topplin' ", "Gettin' ", "requestin' ", "fryin' ", "jackin' ", "Askin' ", "Bustin' ", "snitchin' ", "buildin' ", "buryin' ", "breathtakin' ", "streamlinin' "

In [11]:
# Fix words that end with in'
df['lyrics'] = df['lyrics'].str.replace(
    r"\b(\w+?)in['’]",
    r"\1ing",
    regex=True
)

In [12]:
# Handle vocables
def normalize_vocables(text):
    return re.sub(r'\b(\w{2,})(-\1)+\b', r'\1', text)

df['lyrics'] = df['lyrics'].apply(normalize_vocables)

In [ ]:
# Handle shortened words using mapping
shortened_mapping = {
    "'til": "until",
    "til'": "until",
    "'Til": "Until",
    "'Till": "Until",
    "tryna": "trying to",
    "Tryna": "Trying to",
    "whatchu": "what you",
    "Whatchu": "What you",
    "wit'": "with",
    "fuckin ": "fucking",
    "'bout": "about",
    "'cause": "because",
    "B4": "Before",
    "'em": "them",
    " ya": " you"
}

def replace_shortened_words(text):
    for short, full in shortened_mapping.items():
        text = text.replace(short, full)
    return text

df['lyrics'] = df['lyrics'].apply(replace_shortened_words)

In [14]:
# Remove rows where 'lyrics' is NaN after cleaning
df = df.dropna(subset=["lyrics"])
print("Shape after dropping NaN:", df.shape)

Shape after dropping NaN: (34982, 17)


In [15]:
# Remove duplicate lyrics
df = df.drop_duplicates(subset=['lyrics'], keep='first')

In [ ]:
# Save output
print("After cleaning:", df.shape)
df.to_csv('../preparation/cleanverses.csv', index=False, encoding='utf-8-sig')

After cleaning: (34867, 17)
